# Retention Strategy Simulation

Simulasi ini bertujuan untuk mengevaluasi nilai bisnis dari churn prediction model dengan mengukur potensi Return on Investment (ROI) dari berbagai strategi customer retention. Evaluasi dilakukan dengan membandingkan retention cost (biaya diskon) dengan saved revenue (revenue yang berhasil dipertahankan).

---
## Retention Strategy Simulation Assumptions
Simulasi ROI pada strategi retensi pelanggan dilakukan dengan beberapa asumsi untuk memperkirakan dampak bisnis dari churn prediction model.

### 1. Retention Success Rate
Simulasi menggunakan tiga asumsi tingkat keberhasilan retensi, yaitu **10%, 20%, dan 30%**. Nilai ini merepresentasikan proporsi pelanggan yang berhasil dipertahankan setelah menerima intervensi retensi.

### 2. Discount-Based Retention Program
Program retensi diasumsikan dilakukan dengan memberikan **diskon pada biaya langganan bulanan**. Besaran diskon yang diuji dalam simulasi adalah **5%, 10%, dan 15%**.

### 3. Retention Program Duration
Diskon diberikan dalam dua skenario durasi, yaitu **3 bulan dan 6 bulan**, untuk melihat bagaimana panjang program retensi mempengaruhi biaya dan efektivitas strategi.

### 4. Revenue at Risk Estimation
Potensi revenue yang berisiko hilang diperkirakan berdasarkan **probabilitas churn dari model prediksi** serta estimasi **pendapatan tahunan pelanggan**. Pendapatan tahunan dihitung dengan mengalikan **monthly charges dengan 12 bulan**. Pendekatan ini mengasumsikan bahwa pelanggan yang churn akan berhenti menghasilkan revenue selama satu tahun ke depan.

### 5. Saved Revenue Assumption
Pendapatan yang berhasil diselamatkan diasumsikan berasal dari sebagian pelanggan yang berhasil dipertahankan sesuai dengan **retention rate yang digunakan dalam simulasi**.

### 6. Targeting Strategy
Simulasi menggunakan dua pendekatan penargetan pelanggan, yaitu:
* **High Risk + High Revenue**, yang menargetkan pelanggan dengan churn risk tinggi dan monthly charges tinggi.
* **Top 20% Expected Loss**, yang menargetkan pelanggan dengan potensi kerugian finansial terbesar berdasarkan nilai expected_loss.

### 7. Retention Cost Calculation
Biaya program retensi hanya dihitung berdasarkan **nilai diskon yang diberikan kepada pelanggan selama periode program**, tanpa memasukkan biaya operasional atau marketing lainnya.

### 8. Long-Term Retention Assumption
Pelanggan yang berhasil dipertahankan selama periode diskon diasumsikan **tetap menjadi pelanggan setelah program retensi berakhir**, sehingga revenue mereka dianggap berhasil diselamatkan.

In [1]:
# ====================== Loading Libraries and Data ======================

import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from scr.database import get_engine

engine = get_engine()

# Risk score hasil modeling
df_risk = pd.read_sql("SELECT * FROM churn_risk_score;", engine)

# Dataset original untuk revenue info
df_base = pd.read_sql(
    "SELECT customer_id, monthly_charges FROM churn_feature_dataset;",
    engine
)

df = df_risk.merge(df_base, on="customer_id", how="left")


In [2]:
# ====================== Parameter ======================

RETENTION_RATES = [0.10, 0.20, 0.30]
DISCOUNT_RATES = [0.05, 0.10, 0.15]

RETENTION_MONTHS_OPTIONS = [3, 6]   # diskon berlaku 3 bulan atau 6 bulan

ANNUAL_FACTOR = 12
TOP_PERCENTAGE = 0.20
HIGH_VALUE_THRESHOLD = 70

df["annual_revenue"] = df["monthly_charges"] * ANNUAL_FACTOR

In [3]:
# ====================== Strategy Definition ======================

# Strategy 1: High Risk + High Revenue
strategy_1 = df.loc[
    (df["risk_level"] == "High") &
    (df["monthly_charges"] > HIGH_VALUE_THRESHOLD)
].copy()

# Strategy 2: Top 20% Expected Loss
strategy_2 = df.sort_values(
    "expected_loss", ascending=False
).head(int(len(df) * TOP_PERCENTAGE)).copy()

In [4]:
def simulate_roi(df_segment, discount_rate, retention_rate, discount_months):

    # Cost: diskon hanya untuk beberapa bulan
    total_cost = (
        df_segment["monthly_charges"]
        * discount_months
        * discount_rate
    ).sum()

    # Revenue yang benar-benar berisiko (1 tahun)
    revenue_at_risk = df_segment["expected_loss"]

    # Revenue yang berhasil diselamatkan
    saved_revenue = (revenue_at_risk * retention_rate).sum()

    roi = (saved_revenue - total_cost) / total_cost

    return total_cost, saved_revenue, roi

In [5]:
revenue_at_risk = (
        df["annual_revenue"]
        * df["churn_probability"]
    )

In [6]:
results = []

strategies = {
    "High Risk + High Revenue": strategy_1,
    "Top 20% Expected Loss": strategy_2
}

for strategy_name, df_segment in strategies.items():
    
    for months in RETENTION_MONTHS_OPTIONS:
        
        for discount in DISCOUNT_RATES:
            
            for retention in RETENTION_RATES:
                
                total_cost, saved_revenue, roi = simulate_roi(
                    df_segment,
                    discount,
                    retention,
                    months
                )
                
                results.append({
                    "Strategy": strategy_name,
                    "Discount Months": months,
                    "Discount Rate": f"{int(discount*100)}%",
                    "Retention Rate": f"{int(retention*100)}%",
                    "Target Customers": len(df_segment),
                    "Total Cost": round(total_cost, 2),
                    "Saved Revenue": round(saved_revenue, 2),
                    "ROI": round(roi, 2)
                })

In [7]:
df_simulation = pd.DataFrame(results)

df_simulation_sorted = (
    df_simulation
    .sort_values("ROI", ascending=False)
    .reset_index(drop=True)
)

df_simulation_sorted


,Strategy,Discount Months,Discount Rate,Retention Rate,Target Customers,Total Cost,Saved Revenue,ROI
0,High Risk + High Revenue,3,5%,30%,702,9024.26,152819.41,15.93
1,Top 20% Expected Loss,3,5%,30%,1408,18565.78,272070.46,13.65
2,High Risk + High Revenue,3,5%,20%,702,9024.26,101879.61,10.29
3,Top 20% Expected Loss,3,5%,20%,1408,18565.78,181380.30,8.77
4,High Risk + High Revenue,6,5%,30%,702,18048.51,152819.41,7.47
5,High Risk + High Revenue,3,10%,30%,702,18048.51,152819.41,7.47
6,Top 20% Expected Loss,3,10%,30%,1408,37131.56,272070.46,6.33
7,Top 20% Expected Loss,6,5%,30%,1408,37131.56,272070.46,6.33
8,High Risk + High Revenue,3,10%,20%,702,18048.51,101879.61,4.64
9,High Risk + High Revenue,3,5%,10%,702,9024.26,50939.80,4.64


In [8]:
pivot_roi = df_simulation.pivot_table(
    values="ROI",
    index=["Strategy", "Discount Months", "Discount Rate"],
    columns="Retention Rate"
)

pivot_roi

Retention Rate                                           10%    20%    30%
Strategy                 Discount Months Discount Rate                    
High Risk + High Revenue 3               10%            1.82   4.64   7.47
                                         15%            0.88   2.76   4.64
                                         5%             4.64  10.29  15.93
                         6               10%            0.41   1.82   3.23
                                         15%           -0.06   0.88   1.82
                                         5%             1.82   4.64   7.47
Top 20% Expected Loss    3               10%            1.44   3.88   6.33
                                         15%            0.63   2.26   3.88
                                         5%             3.88   8.77  13.65
                         6               10%            0.22   1.44   2.66
                                         15%           -0.19   0.63   1.44
                                         5%             1.44   3.88   6.33

In [11]:
df_simulation_sorted = df_simulation_sorted.rename(columns={
    "Strategy": "strategy",
    "Discount Months": "discount_months",
    "Discount Rate": "discount_rate",
    "Retention Rate": "retention_rate",
    "Target Customers": "target_customers",
    "Total Cost": "total_cost",
    "Saved Revenue": "saved_revenue",
    "ROI": "roi"
})

In [13]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE retention_simulation_results"))
    conn.commit()

df_simulation_sorted.to_sql(
    "retention_simulation_results",
    engine,
    if_exists="append",
    index=False
)

print("Table retention_simulation_results updated")

Table retention_simulation_results updated


## Retention Strategy Simulation Insights

### 1. Best ROI Scenario

Simulasi menunjukkan bahwa strategi **High Risk + High Revenue** dengan **diskon 5% selama 3 bulan** memberikan ROI tertinggi.

**Parameter**

* Target: **702 customers**
* Discount: **5%**
* Duration: **3 months**
* Assumed retention success: **30%**

**Result**

* Total Cost: **$9,024**
* Saved Revenue: **$152,819**
* **ROI: 15.93**

**Insight**

Strategi ini sangat efisien karena:

* Menargetkan **pelanggan bernilai tinggi dengan risiko churn tinggi**
* Biaya intervensi relatif kecil dibanding revenue yang berpotensi hilang
* Diskon **tidak terlalu agresif**, sehingga biaya tetap terkendali

Namun perlu dicatat bahwa **retention rate 30% cukup optimistis**, sehingga perlu dianalisis juga skenario yang lebih realistis.

---

### 2. More Realistic Business Scenario

Dalam praktik bisnis, **retention campaign biasanya hanya berhasil mempertahankan sekitar 10–20% pelanggan**. Dengan asumsi ini, strategi tetap menghasilkan ROI positif.

Contoh skenario yang lebih realistis:

| Strategy                 | Discount | Duration | Retention | ROI       |
| ------------------------ | -------- | -------- | --------- | --------- |
| High Risk + High Revenue | 5%       | 3 months | 20%       | **10.29** |
| Top 20% Expected Loss    | 5%       | 3 months | 20%       | **8.77**  |

**Insight**

Bahkan pada **retention rate 20%**, program diskon masih menghasilkan ROI yang sangat tinggi (>8x).
Hal ini menunjukkan bahwa **mengintervensi pelanggan sebelum churn secara finansial sangat menguntungkan**.

---

### 3. Strategy Comparison

Dua strategi yang diuji menunjukkan trade-off antara **target size** dan **ROI efficiency**.

| Strategy                 | Target Customers | Strength                                 |
| ------------------------ | ---------------- | ---------------------------------------- |
| High Risk + High Revenue | 702              | ROI lebih tinggi dan biaya lebih efisien |
| Top 20% Expected Loss    | 1408             | Dampak lebih luas terhadap total revenue |

**Insight**

* **High Risk + High Revenue** lebih cocok jika perusahaan ingin **program retensi yang efisien dengan budget kecil**.
* **Top 20% Expected Loss** lebih cocok jika tujuan perusahaan adalah **mengurangi total churn revenue secara besar-besaran**, meskipun ROI sedikit lebih rendah.

---

### 4. Impact of Discount Size

Simulasi menunjukkan pola yang cukup konsisten:

* **Diskon kecil (5%) memberikan ROI terbaik**
* **Diskon besar (15%) cenderung menurunkan ROI**

Contoh:

| Discount | Duration | ROI Range        |
| -------- | -------- | ---------------- |
| 5%       | 3 months | **4.64 – 15.93** |
| 10%      | 3 months | 1.44 – 7.47      |
| 15%      | 3 months | 0.63 – 4.64      |

**Insight**

Memberikan diskon terlalu besar **tidak selalu meningkatkan efektivitas retensi**, tetapi justru dapat **menggerus profit margin**.

---
### 5. Retention Program Duration

Program diskon **3 bulan** secara konsisten menghasilkan **ROI lebih tinggi dibandingkan 6 bulan**.

Contoh:

| Program Duration | ROI       |
| ---------------- | --------- |
| 3 months         | **15.93** |
| 6 months         | 7.47      |

**Insight**

* Hal ini terjadi karena **retention cost meningkat secara linear dengan durasi diskon**, sementara **revenue yang diselamatkan tidak meningkat secara langsung**.
* Dari perspektif bisnis, **short-term retention intervention** lebih efisien dibandingkan program diskon jangka panjang.

---
### 6. Recommended Retention Strategy

Berdasarkan simulasi ROI dan pertimbangan bisnis:

**Recommended campaign design**

* Target: **High Risk + High Revenue customers**
* Discount: **5%**
* Duration: **3 months**
* Expected retention: **10–20%**

**Expected impact**

* Program tetap menghasilkan **ROI antara 4.6 – 10.3**
* Biaya kampanye relatif kecil
* Risiko kerugian dari diskon berlebihan dapat dihindari